In [ ]:
# ==========================================
# NOTEBOOK 2: DOMINIOS DERIVADOS
# ==========================================

import sys
from pathlib import Path

# notebook/jupyter → proyecto/
PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from scr.python.clase_datalake import (
    ParquetStore,
    # Funciones de transformación
    euros_pais,
    euros_sectores,
    euros_sectores_pais,
    euros_taric,
    euros_taric_pais,
    kg_pais,
    kg_taric,
    kg_taric_pais,
)

%load_ext autoreload
%autoreload 2

# ==========================================
# CONFIGURACIÓN
# ==========================================

DATA_LAKE_DIR = "/home/pivan/comercio_exterior_bienes/data/rawparquet"

# Rangos temporales
ULTIMO_ANIO_DEF = 2023
ULTIMO_ANIO_PROV = 2025
ULTIMO_MES_DISPONIBLE = 10

YEARS_DEF = (1995, ULTIMO_ANIO_DEF)
YEARS_PROV = (2024, ULTIMO_ANIO_PROV)

# Ámbitos a procesar
AMBITOS = ["espana", "madrid"]

# ==========================================
# DEFINICIÓN DEL PIPELINE
# ==========================================

# Cada tupla es: (dominio_entrada, dominio_salida, función_transformación)
PIPELINE = [
    # -------- TARIC - Euros
    ("taric", "euros_taric",       euros_taric),
    ("taric", "euros_taric_pais",  euros_taric_pais),
    ("taric", "euros_pais",        euros_pais),
    
    # -------- TARIC - Kilogramos
    ("taric", "kg_taric",          kg_taric),
    ("taric", "kg_taric_pais",     kg_taric_pais),
    ("taric", "kg_pais",           kg_pais),

    # -------- SECTORES - Euros
    ("sectores", "euros_sectores",       euros_sectores),
    ("sectores", "euros_sectores_pais",  euros_sectores_pais),
]

# ==========================================
# INICIALIZAR STORE
# ==========================================

store = ParquetStore(DATA_LAKE_DIR)

print("=== Pipeline de Dominios Derivados ===")
print(f"Data Lake: {DATA_LAKE_DIR}")
print(f"Transformaciones: {len(PIPELINE)}")
print(f"Ámbitos: {', '.join(AMBITOS)}")
print()

# ==========================================
# EJECUCIÓN AUTOMÁTICA
# ==========================================

# Opción 1: Ejecutar pipeline completo con el método integrado
# -------------------------------------------------------------

# Datos provisionales (estado=0)
stats_prov = store.process_derived_pipeline(
    pipeline=PIPELINE,
    ambitos=AMBITOS,
    year_start=YEARS_PROV[0],
    year_end=YEARS_PROV[1],
    estado=0,  # provisional
    ultimo_mes=ULTIMO_MES_DISPONIBLE,
    skip_if_exists=True
)

# Datos definitivos (estado=1)
stats_def = store.process_derived_pipeline(
    pipeline=PIPELINE,
    ambitos=AMBITOS,
    year_start=YEARS_DEF[0],
    year_end=YEARS_DEF[1],
    estado=1,  # definitivo
    skip_if_exists=True
)

# Mostrar resumen
print("\n" + "="*60)
print("RESUMEN FINAL")
print("="*60)

print(f"\nDatos provisionales ({YEARS_PROV[0]}-{YEARS_PROV[1]}):")
print(f"  • Procesados: {stats_prov['procesados']}")
print(f"  • Saltados: {stats_prov['saltados']}")
print(f"  • Errores: {stats_prov['errores']}")

print(f"\nDatos definitivos ({YEARS_DEF[0]}-{YEARS_DEF[1]}):")
print(f"  • Procesados: {stats_def['procesados']}")
print(f"  • Saltados: {stats_def['saltados']}")
print(f"  • Errores: {stats_def['errores']}")

print("\n✅ Pipeline completado")


# ==========================================
# Opción 2: Procesamiento manual (más control)
# ==========================================
# Útil para debugging o casos específicos

# # Procesar un dominio específico
# store.process_derived_domain(
#     ambito="madrid",
#     dominio_in="taric",
#     dominio_out="euros_taric",
#     estado=1,  # definitivo
#     anio=2024,
#     mes=1,
#     transform=euros_taric,
#     skip_if_exists=False  # Forzar reprocesamiento
# )


# ==========================================
# VERIFICACIÓN Y ANÁLISIS
# ==========================================

print("\n" + "="*60)
print("VERIFICACIÓN DE DOMINIOS DERIVADOS")
print("="*60)

# Verificar algunos dominios clave
YEAR_TEST = 2024
MONTH_TEST = 1

for ambito in AMBITOS:
    print(f"\n{ambito.upper()}:")
    for _, dominio_out, _ in PIPELINE[:3]:  # Verificar primeros 3
        exists = store.exists(ambito, dominio_out, estado=0, anio=YEAR_TEST, mes=MONTH_TEST)
        status = "✅" if exists else "❌"
        print(f"  {status} {dominio_out}")

# ==========================================
# ANÁLISIS RÁPIDO
# ==========================================
# Cargar y mostrar datos de ejemplo

print("\n" + "="*60)
print("ANÁLISIS DE DATOS")
print("="*60)

# Cargar datos de España - euros_taric
df = store.read(
    ambito="espana",
    dominio="euros_taric",
    estado=1,  # definitivo
    anio=2023,
    mes=12
)

print(f"\n📊 España - euros_taric (2023-12):")
print(f"  Registros: {len(df):,}")
print(f"  Columnas: {', '.join(df.columns)}")
print(f"\n  Top 5 exportaciones (nivel 2):")

top_exp = (
    df
    .filter((df["flujo"] == 1) & (df["nivel_taric"] == 2))
    .sort("euros", descending=True)
    .head(5)
)
print(top_exp)

print(f"\n  Total exportaciones: €{df.filter(df['flujo'] == 1)['euros'].sum():,.0f}")
print(f"  Total importaciones: €{df.filter(df['flujo'] == 0)['euros'].sum():,.0f}")


# ==========================================
# UTILIDADES ADICIONALES
# ==========================================

# Función helper para verificar disponibilidad
def verificar_disponibilidad(ambito: str, dominio: str, estado: int, anio: int):
    """Verifica qué meses están disponibles para un año."""
    meses_disponibles = []
    for mes in range(1, 13):
        if store.exists(ambito, dominio, estado, anio, mes):
            meses_disponibles.append(mes)
    return meses_disponibles

# Ejemplo de uso
meses = verificar_disponibilidad("madrid", "euros_taric", estado=1, anio=2023)
print(f"\n📅 Meses disponibles para Madrid - euros_taric (2023): {meses}")